In [1]:
import numpy as np
import pandas as pd

# Keep every generated synthetic table reproducible.
RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

# These are transaction/service codes, not physical merchandise.
NON_MERCHANDISE_CODES = {
    "ADJUST", "ADJUST2", "AMAZONFEE", "B", "BANK CHARGES",
    "C2", "CRUK", "D", "DOT", "M", "POST", "S"
}

# Resolve paths from the project root or the scripts/ directory.
from pathlib import Path
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "scripts":
    PROJECT_DIR = PROJECT_DIR.parent
DATA_DIR = PROJECT_DIR / "data"

df = pd.read_csv(DATA_DIR / "online_retail.csv")
df.head()


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [2]:
df.info()
df.isnull().sum()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  object 
 1   StockCode    1067371 non-null  object 
 2   Description  1062989 non-null  object 
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  object 
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  object 
dtypes: float64(2), int64(1), object(5)
memory usage: 65.1+ MB


,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


Cleaning and standardising the source data. Missing customer IDs are assigned to the 999999 unknown-customer bucket; returns are retained; rows with non-positive unit prices are excluded; product codes are trimmed.

In [3]:
df['Customer ID'] = df['Customer ID'].fillna(999999).astype(int)
df['Description'] = df['Description'].fillna('Unknown Product')

# Trim accidental whitespace so one StockCode maps to one product.
df['StockCode'] = df['StockCode'].astype('string').str.strip()

# Keep the existing positive-price quality filter. Returns remain in the data,
# including returns whose customer ID is unknown (999999).
df = df[df['Price'] > 0].copy()

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], format='%Y-%m-%d %H:%M:%S')


In [4]:
df.shape
df.isnull().sum()
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.061164e+06,1061164,1.061164e+06,1.061164e+06
mean,1.029936e+01,2011-01-03 06:21:20.383220224,4.826113e+00,2.351218e+05
min,-8.099500e+04,2009-12-01 07:45:00,1.000000e-03,1.234600e+04
25%,1.000000e+00,2010-07-09 15:48:00,1.250000e+00,1.434400e+04
50%,3.000000e+00,2010-12-07 15:31:00,2.100000e+00,1.615000e+04
75%,1.000000e+01,2011-07-22 14:53:00,4.150000e+00,1.802200e+04
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,9.999990e+05
std,1.662605e+02,NaN,9.571170e+01,4.100246e+05


Extracting the Products Table

In [5]:
# Use the median price so one-off price spikes do not inflate synthetic costs.
df_prod = df.groupby('StockCode').aggregate({'Price' : 'median'})

df_prod = df_prod.reset_index()

df_prod.head()


,StockCode,Price
0,10002,0.85
1,10002R,5.17
2,10080,0.39
3,10109,0.42
4,10120,0.21


In [6]:
uniq_prod = df[['StockCode', 'Description']]

uniq_prod = uniq_prod.drop_duplicates(subset=['StockCode'])

uniq_prod.head()

,StockCode,Description
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323P,PINK CHERRY LIGHTS
2,79323W,WHITE CHERRY LIGHTS
3,22041,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,STRAWBERRY CERAMIC TRINKET BOX


In [7]:
df_prod = df_prod.merge(uniq_prod, how='left', on='StockCode')

df_prod.head()

,StockCode,Price,Description
0,10002,0.85,INFLATABLE POLITICAL GLOBE
1,10002R,5.17,ROBOT PENCIL SHARPNER
2,10080,0.39,GROOVY CACTUS INFLATABLE
3,10109,0.42,BENDY COLOUR PENCILS
4,10120,0.21,DOGGY RUBBER


In [8]:
df_prod['unit_cost'] = (
    rng.uniform(low=0.4, high=0.7, size=len(df_prod)) * df_prod['Price']
).round(2)

# Synthetic unit costs are only assigned to physical merchandise.
product_code_upper = df_prod['StockCode'].str.upper()
non_merchandise_mask = (
    product_code_upper.isin(NON_MERCHANDISE_CODES)
    | product_code_upper.str.startswith('GIFT_0001_', na=False)
)
df_prod.loc[non_merchandise_mask, 'unit_cost'] = np.nan


In [9]:
df_prod.head()

,StockCode,Price,Description,unit_cost
0,10002,0.85,INFLATABLE POLITICAL GLOBE,0.54
1,10002R,5.17,ROBOT PENCIL SHARPNER,2.75
2,10080,0.39,GROOVY CACTUS INFLATABLE,0.26
3,10109,0.42,BENDY COLOUR PENCILS,0.26
4,10120,0.21,DOGGY RUBBER,0.09


In [10]:
df_prod.rename(columns={'StockCode': 'product_id', 'Description': 'product_name'}, inplace=True)

df_prod.drop(columns = ['Price'], inplace=True)

df_prod.head(10)

,product_id,product_name,unit_cost
0,10002,INFLATABLE POLITICAL GLOBE,0.54
1,10002R,ROBOT PENCIL SHARPNER,2.75
2,10080,GROOVY CACTUS INFLATABLE,0.26
3,10109,BENDY COLOUR PENCILS,0.26
4,10120,DOGGY RUBBER,0.09
5,10123C,HEARTS WRAPPING TAPE,0.45
6,10123G,ARMY CAMO WRAPPING TAPE,0.41
7,10124A,SPOTS ON RED BOOKCOVER TAPE,0.27
8,10124G,ARMY CAMO BOOKCOVER TAPE,0.18
9,10125,MINI FUNKY DESIGN TAPES,0.45


Extracting the Orders Table & Saving to CSV

In [11]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


In [12]:
df_order = df.rename(columns={'Invoice': 'order_id', 'Customer ID': 'customer_id', 'InvoiceDate': 'order_date', 'StockCode': 'product_id', 'Quantity': 'quantity', 'Price': 'unit_selling_price', 'Country': 'region'})

df_order = df_order.drop(columns=['Description'], inplace=False)

# A stable row-level key is needed because order_id + product_id is not unique.
df_order.insert(0, 'order_line_id', np.arange(1, len(df_order) + 1, dtype=np.int64))

df_order.head()


,order_line_id,order_id,product_id,quantity,order_date,unit_selling_price,customer_id,region
0,1,489434,85048,12,2009-12-01 07:45:00,6.95,13085,United Kingdom
1,2,489434,79323P,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
2,3,489434,79323W,12,2009-12-01 07:45:00,6.75,13085,United Kingdom
3,4,489434,22041,48,2009-12-01 07:45:00,2.10,13085,United Kingdom
4,5,489434,21232,24,2009-12-01 07:45:00,1.25,13085,United Kingdom


Save both products_df and orders_df

In [13]:
df_prod.to_csv(DATA_DIR / 'products.csv', index=False)
df_order.to_csv(DATA_DIR / 'orders.csv', index=False)

Generating the Invoices Table before we tackle the complex payment behaviors

In [14]:
df_order['line_total'] = df_order['quantity'] * df_order['unit_selling_price']

df_order.head()

,order_line_id,order_id,product_id,quantity,order_date,unit_selling_price,customer_id,region,line_total
0,1,489434,85048,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,83.4
1,2,489434,79323P,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
2,3,489434,79323W,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,81.0
3,4,489434,22041,48,2009-12-01 07:45:00,2.10,13085,United Kingdom,100.8
4,5,489434,21232,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,30.0


In [15]:
df_invoice = df_order.groupby(['order_id', 'customer_id']).agg({'line_total': 'sum', 'order_date': 'min'}).reset_index()

df_invoice = df_invoice.rename(columns={'line_total': 'invoice_amount', 'order_date': 'invoice_date'})

# Preserve the source's three-decimal unit-price precision and remove float noise.
df_invoice['invoice_amount'] = df_invoice['invoice_amount'].round(3)

df_invoice.head()

,order_id,customer_id,invoice_amount,invoice_date
0,489434,13085,505.30,2009-12-01 07:45:00
1,489435,13085,145.80,2009-12-01 07:46:00
2,489436,13078,630.33,2009-12-01 09:06:00
3,489437,15362,310.75,2009-12-01 09:08:00
4,489438,18102,2286.24,2009-12-01 09:24:00


In [16]:
df_invoice['invoice_id'] =  'INV-' + df_invoice['order_id'].astype(str)


In [17]:
df_invoice['Due_Date'] = df_invoice['invoice_date'] + pd.to_timedelta(30, unit='D')

df_invoice.drop(columns = ['order_id'], inplace=True)


In [18]:
df_invoice.rename(columns={'Due_Date': 'due_date'}, inplace=True)

df_invoice = df_invoice[['invoice_id', 'customer_id', 'invoice_date', 'due_date', 'invoice_amount']]

df_invoice.shape

df_invoice.head()


,invoice_id,customer_id,invoice_date,due_date,invoice_amount
0,INV-489434,13085,2009-12-01 07:45:00,2009-12-31 07:45:00,505.30
1,INV-489435,13085,2009-12-01 07:46:00,2009-12-31 07:46:00,145.80
2,INV-489436,13078,2009-12-01 09:06:00,2009-12-31 09:06:00,630.33
3,INV-489437,15362,2009-12-01 09:08:00,2009-12-31 09:08:00,310.75
4,INV-489438,18102,2009-12-01 09:24:00,2009-12-31 09:24:00,2286.24


In [19]:
(df_invoice['invoice_amount'] <= 0).sum()

np.int64(8291)

In [20]:
(df_invoice['invoice_id'].str.startswith('INV-C') & (df_invoice['invoice_amount'] < 0)).sum()

np.int64(8291)

In [21]:
(df_invoice['invoice_id'].str.startswith('INV-C') & (df_invoice['invoice_amount'] > 0)).sum()

np.int64(1)

In [22]:
df_invoice.loc[df_invoice['invoice_id'].str.startswith('INV-C') & (df_invoice['invoice_amount'] > 0)].head()

,invoice_id,customer_id,invoice_date,due_date,invoice_amount
40778,INV-C496350,999999,2010-02-01 08:24:00,2010-03-03 08:24:00,373.57


In [23]:
df_order[df_order['order_id'] == 'C496350']

,order_line_id,order_id,product_id,quantity,order_date,unit_selling_price,customer_id,region,line_total
76799,76313,C496350,M,1,2010-02-01 08:24:00,373.57,999999,United Kingdom,373.57


In [24]:
# 1. Separate C-prefixed credit/cancellation-style transactions
credit_notes_df = df_invoice[
    df_invoice['invoice_id'].str.startswith('INV-C')
].copy()

# 2. Keep positive, non-C-prefixed invoices for payment simulation
invoices_df = df_invoice[
    (~df_invoice['invoice_id'].str.startswith('INV-C')) &
    (df_invoice['invoice_amount'] > 0)
].copy()

# 3. Verify the results
print("Total invoices:", len(df_invoice))
print("Credit/cancellation-style records:", len(credit_notes_df))
print("Invoices eligible for payment simulation:", len(invoices_df))

print(invoices_df.head())

Total invoices: 48369
Credit/cancellation-style records: 8292
Invoices eligible for payment simulation: 40077
   invoice_id  customer_id  ...            due_date invoice_amount
0  INV-489434        13085  ... 2009-12-31 07:45:00         505.30
1  INV-489435        13085  ... 2009-12-31 07:46:00         145.80
2  INV-489436        13078  ... 2009-12-31 09:06:00         630.33
3  INV-489437        15362  ... 2009-12-31 09:08:00         310.75
4  INV-489438        18102  ... 2009-12-31 09:24:00        2286.24

[5 rows x 5 columns]


Generating the Payments Table.

In [25]:
invoices_df['payment_status'] = rng.choice(
    ['paid_on_time', 'paid_late', 'partial_payment', 'unpaid'],
    size=len(invoices_df),
    p=[0.6, 0.2, 0.1, 0.1]
)


In [26]:
invoices_df['payment_status'].value_counts()

payment_status
paid_on_time       23831
paid_late           8240
unpaid              4014
partial_payment     3992
Name: count, dtype: int64

In [27]:
payment_df = invoices_df[['invoice_id', 'customer_id', 'invoice_date', 'due_date', 'invoice_amount', 'payment_status']].copy()

payment_df = payment_df[payment_df['payment_status'] != 'unpaid']

In [28]:
payment_df['payment_status'].value_counts()

payment_status
paid_on_time       23831
paid_late           8240
partial_payment     3992
Name: count, dtype: int64

In [29]:
payment_df['payment_date'] = payment_df.apply(
    lambda row: (
        row['invoice_date'] + pd.to_timedelta(
            rng.integers(0, 31), unit='D'
        )
        if row['payment_status'] in ['paid_on_time', 'partial_payment']
        else row['due_date'] + pd.to_timedelta(
            rng.integers(5, 61), unit='D'
        )
    ),
    axis=1
)


In [30]:
payment_df.head()

,invoice_id,customer_id,invoice_date,due_date,invoice_amount,payment_status,payment_date
0,INV-489434,13085,2009-12-01 07:45:00,2009-12-31 07:45:00,505.30,paid_late,2010-01-07 07:45:00
1,INV-489435,13085,2009-12-01 07:46:00,2009-12-31 07:46:00,145.80,paid_late,2010-02-18 07:46:00
2,INV-489436,13078,2009-12-01 09:06:00,2009-12-31 09:06:00,630.33,paid_on_time,2009-12-18 09:06:00
3,INV-489437,15362,2009-12-01 09:08:00,2009-12-31 09:08:00,310.75,paid_late,2010-01-18 09:08:00
4,INV-489438,18102,2009-12-01 09:24:00,2009-12-31 09:24:00,2286.24,paid_on_time,2009-12-17 09:24:00


In [31]:
payment_df['payment_amount'] = payment_df.apply(
    lambda row: (
        round(row['invoice_amount'] * rng.uniform(0.1, 0.9), 3)
        if row['payment_status'] == 'partial_payment'
        else round(row['invoice_amount'], 3)
    ),
    axis=1
)


In [32]:
payment_df['payment_id'] = 'PAY-' + payment_df['invoice_id']

In [33]:
payment_df['payment_id'].is_unique
payment_df['payment_id'].isna().sum()

np.int64(0)

In [34]:
payment_df = payment_df[
    ['payment_id', 'invoice_id', 'payment_date', 'payment_amount']
]

In [35]:
payment_df.head()

,payment_id,invoice_id,payment_date,payment_amount
0,PAY-INV-489434,INV-489434,2010-01-07 07:45:00,505.30
1,PAY-INV-489435,INV-489435,2010-02-18 07:46:00,145.80
2,PAY-INV-489436,INV-489436,2009-12-18 09:06:00,630.33
3,PAY-INV-489437,INV-489437,2010-01-18 09:08:00,310.75
4,PAY-INV-489438,INV-489438,2009-12-17 09:24:00,2286.24


In [36]:
print(payment_df.shape)
print(payment_df.columns.tolist())
print(payment_df['payment_id'].is_unique)
print(payment_df.isna().sum())

(36063, 4)
['payment_id', 'invoice_id', 'payment_date', 'payment_amount']
True
payment_id        0
invoice_id        0
payment_date      0
payment_amount    0
dtype: int64


Generating the Inventory Snapshots Table

In [37]:
# Generate month-end inventory snapshots over the analysis period.
snapshot_dates = pd.date_range(
    start='2009-12-31',
    end='2011-12-31',
    freq='ME'
)

# Inventory is generated only for merchandise with a defined synthetic cost.
merchandise_product_ids = df_prod.loc[
    df_prod['unit_cost'].notna(), 'product_id'
].unique()

inventory_df = pd.MultiIndex.from_product(
    [merchandise_product_ids, snapshot_dates],
    names=['product_id', 'snapshot_date']
).to_frame(index=False)

# Synthetic month-end stock quantities (fixed seed above makes them repeatable).
inventory_df['stock_quantity'] = rng.integers(
    0, 500, size=len(inventory_df)
)

print(inventory_df.head())
print(inventory_df.shape)
print(inventory_df.info())


  product_id snapshot_date  stock_quantity
0      10002    2009-12-31             325
1      10002    2010-01-31             199
2      10002    2010-02-28             412
3      10002    2010-03-31              37
4      10002    2010-04-30             234
(122775, 3)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122775 entries, 0 to 122774
Data columns (total 3 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   product_id      122775 non-null  string        
 1   snapshot_date   122775 non-null  datetime64[ns]
 2   stock_quantity  122775 non-null  int64         
dtypes: datetime64[ns](1), int64(1), string(1)
memory usage: 2.8 MB
None


In [38]:
invoices_df.to_csv(DATA_DIR / 'invoices.csv', index=False)
payment_df.to_csv(DATA_DIR / 'payments.csv', index=False)
inventory_df.to_csv(DATA_DIR / 'inventory.csv', index=False)

In [39]:
invoices_df = invoices_df.drop(columns=['payment_status'])
invoices_df.to_csv(DATA_DIR / 'invoices.csv', index=False)